In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
.appName("PySparkPractice") \
.getOrCreate()

In [3]:
products_data = [
     ("P01", "Laptop", "Electronics", 60000),
     ("P02", "Monitor", "Electronics", 14000),
     ("P03", "Headphones", "Accessories", 3500),
     ("P04", "Printer", "Office", 17000),
     ("P05", "Router", "Networking", 4500),
     ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
products_data,
     ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
stores_data, ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+



In [4]:
sales_base_df = (
  spark.range(1, 100001)
  .withColumnRenamed("id", "order_id")
  .withColumn(
     "order_date",
     F.date_add(
    F.to_date(F.lit("2026-01-01")),(F.col("order_id") % 180).cast("int")
        )
     )
.withColumn(
    "product_id",
     F.concat(
     F.lit("P"),
     F.lpad(((F.col("order_id") % 6) + 1).cast("string"),
      2, "0"
     )
   )
)
.withColumn(
   "store_id",
    F.concat(
    F.lit("S"),
    F.lpad(((F.floor(F.col("order_id") / 3) % 4) + 1).cast("string"),
   2, "0"
    )
  )
)
.withColumn(
   "quantity",
  ((F.col("order_id") % 8) + 1).cast("int")
)
.withColumn(
   "discount_pct",
    F.when(
    F.col("order_id") % 19 == 0,
    F.lit(None).cast("int")
).otherwise(
  ((F.floor(F.col("order_id") / 5) % 4) * 5)
  .cast("int")
  )
 )
)

In [5]:
sales_df = (
    sales_base_df
    .withColumn(
         "order_status",
    F.when(F.col("order_id") % 11 == 0, "Cancelled")
   .when(F.col("order_id") % 7 == 0, "Returned")
   .otherwise("Completed")
)
.withColumn(
    "payment_method",
F.when(F.col("order_id") % 3 == 0, "UPI")
.when(F.col("order_id") % 3 == 1, "Card")
.otherwise("Cash")
)
.withColumn(
    "salesperson_id",
  F.when(
  F.col("order_id") % 17 == 0,
  F.lit(None).cast("string")
).otherwise(
   F.concat(
   F.lit("EMP"),
  F.lpad(((F.col("order_id") % 12) + 1).cast("string"),
2, "0"
    )
  )
 )
)
.join(
products_df.select(
   "product_id",
  F.col("list_price").alias("unit_price")
),
  "product_id", "left"
)
.select(
   "order_id", "order_date", "store_id",
   "product_id", "quantity", "unit_price",
  "discount_pct", "order_status",
  "payment_method", "salesperson_id"
 )
)
print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |C

In [6]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show(5)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       2|       P03|       3|      3500|
|       4|       P05|       5|      4500|
|       5|       P06|       6|     24000|
|       1|       P02|       2|     14000|
+--------+----------+--------+----------+
only showing top 5 rows


In [7]:
sales_df.filter((F.col("order_status") == "Completed") & (F.col("quantity") > 5)).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      23|2026-01-24|     S04|       P06|       8|     24000|           0|   Completed|          Cash|         EMP12|
|      29|2026-01-30|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      13|2026-01-14|     S01|       P02|       6|     14000|          10|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [8]:
sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|        10500|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|        22500|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|       144000|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|  

In [10]:
sales_df.withColumnRenamed("order_status", "transaction_status").drop("payment_method").show(5)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|         Completed|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|         Completed|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|         Completed|         EMP02|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
only showing top 5 

In [11]:
sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity")).show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P04|         83334|
|       P03|         66667|
|       P05|         66665|
|       P06|         83330|
|       P02|         83336|
|       P01|         66668|
+----------+--------------+



In [12]:
sales_df \
    .join(products_df, "product_id", "inner") \
    .groupBy("category") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("total_gross_revenue")) \
    .show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|     Office|         1416678000|
| Networking|          299992500|
|Electronics|         7166704000|
|Accessories|          233334500|
+-----------+-------------------+



In [13]:
sales_df \
    .join(stores_df, "store_id", "inner") \
    .groupBy("city") \
    .agg(F.count("order_id").alias("total_transactions")) \
    .show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|   Mumbai|             24999|
|    Delhi|             24999|
|Bengaluru|             25001|
|Hyderabad|             25001|
+---------+------------------+



In [14]:
sales_df \
    .filter(F.col("order_status") == "Completed") \
    .join(products_df, "product_id", "inner") \
    .groupBy("product_name") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("gross_revenue")) \
    .orderBy(F.col("gross_revenue").desc()) \
    .limit(5) \
    .show()

+------------+-------------+
|product_name|gross_revenue|
+------------+-------------+
|      Laptop|   3116940000|
|      Tablet|   1558416000|
|     Printer|   1103912000|
|     Monitor|    909076000|
|      Router|    233766000|
+------------+-------------+



In [15]:
sales_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("null_discount_pct"),
    F.count(F.when(F.col("salesperson_id").isNull(), 1)).alias("null_salesperson_id")
).show()

+-----------------+-------------------+
|null_discount_pct|null_salesperson_id|
+-----------------+-------------------+
|             5263|               5882|
+-----------------+-------------------+



In [16]:
sales_df.na.fill({"discount_pct": 0}).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [17]:
sales_df.na.drop(subset=["salesperson_id"]).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [18]:
sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity").between(1, 3), "Small")
     .when(F.col("quantity").between(4, 6), "Medium")
     .otherwise("Large")
).show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|    Medium|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|     Small|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|    Medium|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|    Medium|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          

In [19]:
window_spec_rank = Window.partitionBy("category").orderBy(F.col("total_revenue").desc())
sales_df \
    .join(products_df, "product_id", "inner") \
    .groupBy("category", "product_name") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("total_revenue")) \
    .withColumn("rank", F.dense_rank().over(window_spec_rank)) \
    .show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Accessories|  Headphones|    233334500|   1|
|Electronics|      Laptop|   4000080000|   1|
|Electronics|      Tablet|   1999920000|   2|
|Electronics|     Monitor|   1166704000|   3|
| Networking|      Router|    299992500|   1|
|     Office|     Printer|   1416678000|   1|
+-----------+------------+-------------+----+



In [20]:
window_spec_run = Window.partitionBy("store_id").orderBy("order_date", "order_id")
sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
        .withColumn("running_revenue", F.sum("gross_revenue").over(window_spec_run)) \
        .show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|running_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------+
|     180|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|         300000|
|     360|2026-01-01|     S01|       P01|       1|     60000|           0|   Completed|           UPI|         EMP01|        60000|         360000|
|     540|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|         660000|
|     720|2026-01-01|     S01|       P01|       1|     60000|           0|   Completed|           UPI|         E

In [21]:
sales_df \
    .filter(F.col("order_status") == "Completed") \
    .withColumn("month", F.month("order_date")) \
    .groupBy("month") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("monthly_revenue")) \
    .orderBy("month") \
    .show()

+-----+---------------+
|month|monthly_revenue|
+-----+---------------+
|    1|     1278728500|
|    2|     1027585500|
|    3|     1212087500|
|    4|     1218921000|
|    5|     1277560500|
|    6|     1089027500|
+-----+---------------+



In [ ]:
print("Applying filter transformation...")
lazy_step = sales_df.filter(F.col("order_status") == "Completed")
print("Applying withColumn transformation...")
lazy_step = lazy_step.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("Applying select transformation...")
lazy_step = lazy_step.select("order_id", "gross_revenue")
print("No Spark jobs have run yet. Triggering execution now with show():")
lazy_step.show(5)

Applying filter transformation...
Applying withColumn transformation...
Applying select transformation...
No Spark jobs have run yet. Triggering execution now with show():
+--------+-------------+
|order_id|gross_revenue|
+--------+-------------+
|       3|        68000|
|       2|        10500|
|       4|        22500|
|       5|       144000|
|       1|        28000|
+--------+-------------+
only showing top 5 rows


In [23]:
print("=== Physical Plan ===")
sales_df.groupBy("product_id").count().explain()
results = sales_df.groupBy("product_id").count().collect()

=== Physical Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[count(1)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_count(1)])
      +- Project [product_id#33]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4189]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=4190]
                  +- Project [product_id#0]
                     +- Filter isnotnull(product_id#0)
                        +- Scan ExistingRDD[product_id#0,product_name#1,category#2,list_price#3L]


In [24]:
sales_df_completed = sales_df.filter(F.col("order_status") == "Completed").cache()
start_time = time.time()
count1 = sales_df_completed.count()
print(f"First count action completed in: {time.time() - start_time:.4f} seconds (Builds cache)")
start_time = time.time()
count2 = sales_df_completed.count()
print(f"Second count action completed in: {time.time() - start_time:.4f} seconds (Reads from cache)")

First count action completed in: 10.0898 seconds (Builds cache)
Second count action completed in: 1.6129 seconds (Reads from cache)


In [25]:
sales_df_large = sales_df.filter(F.col("quantity") > 5).persist(StorageLevel.MEMORY_AND_DISK)
print("Action 1 (Count):", sales_df_large.count())
print("Action 2 (First Row):", sales_df_large.first())
sales_df_large.unpersist()

Action 1 (Count): 37500
Action 2 (First Row): Row(order_id=15, order_date=datetime.date(2026, 1, 16), store_id='S02', product_id='P04', quantity=8, unit_price=17000, discount_pct=15, order_status='Completed', payment_method='UPI', salesperson_id='EMP04')


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [26]:
sales_df_cached_agg = sales_df.groupBy("store_id").count().cache()
sales_df_cached_agg.count() # Materialize cache
print("=== Cached Query Plan ===")
sales_df_cached_agg.explain()

=== Cached Query Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [store_id#34, count#1670L]
      +- InMemoryRelation [store_id#34, count#1670L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 4
                  +- *(4) HashAggregate(keys=[store_id#34], functions=[count(1)])
                     +- ShuffleQueryStage 3
                        +- Exchange hashpartitioning(store_id#34, 200), ENSURE_REQUIREMENTS, [plan_id=4939]
                           +- *(3) HashAggregate(keys=[store_id#34], functions=[partial_count(1)])
                              +- *(3) Project [store_id#34]
                                 +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                                    :- AQEShuffleRead local
                                    :  +- ShuffleQueryStage 0
     

In [27]:
print("Original partition count:", sales_df.rdd.getNumPartitions())
print("New partition count:", sales_df.repartition(8).rdd.getNumPartitions())

Original partition count: 2
New partition count: 8


In [28]:
sales_df.repartition(8) \
    .groupBy(F.spark_partition_id().alias("partition_id")) \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [29]:
sales_df.repartition(6, "store_id") \
    .groupBy(F.spark_partition_id().alias("partition_id"), "store_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           1|     S03|24999|
|           3|     S02|25001|
|           4|     S01|25001|
|           5|     S04|24999|
+------------+--------+-----+



In [30]:
print("=== Coalesce Physical Plan ===")
sales_df.repartition(12).coalesce(4).explain()
print("=== Repartition Physical Plan ===")
sales_df.repartition(12).repartition(4).explain()

=== Coalesce Physical Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 4
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=5866]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5860]
            :     +- Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, concat(S, lpad(cast(((FLOOR((cast(id#30L as double) / 3.0)) % 4) + 1) as string), 2, 0)) AS store_id#34, cast(((id#30L % 8) + 1) as int) AS quantity#35, CASE WHEN ((id#30L % 19) = 0) THEN null ELSE cast(((FLOOR((cas

In [31]:
sales_df.repartition(4).write.mode("overwrite").csv("sales_output_csv", header=True)

In [32]:
window_spec_challenge = Window.partitionBy("region").orderBy(F.col("total_gross_revenue").desc())
sales_processed = sales_df \
    .filter(F.col("order_status") == "Completed") \
    .na.fill({"discount_pct": 0}) \
    .join(products_df, "product_id", "inner") \
    .join(stores_df, "store_id", "inner") \
    .withColumn("gross_revenue", F.col("quantity") * F.col("list_price")) \
    .withColumn("discounted_revenue", F.col("gross_revenue") * (1 - F.col("discount_pct") / 100)) \
    .groupBy("region", "category") \
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue"),
        F.sum("discounted_revenue").alias("total_discounted_revenue"),
        F.count("order_id").alias("transaction_count")
    ) \
    .withColumn("rank", F.dense_rank().over(window_spec_challenge))


In [33]:
sales_processed.cache()

DataFrame[region: string, category: string, total_gross_revenue: bigint, total_discounted_revenue: double, transaction_count: bigint, rank: int]

In [34]:
sales_processed.orderBy("region", "rank").show(truncate=False)

+------+-----------+-------------------+------------------------+-----------------+----+
|region|category   |total_gross_revenue|total_discounted_revenue|transaction_count|rank|
+------+-----------+-------------------+------------------------+-----------------+----+
|North |Electronics|934848000          |8.551824E8              |6492             |1   |
|North |Office     |441388000          |4.12114E8               |6492             |2   |
|North |Networking |146097000          |1.35026775E8            |6494             |3   |
|South |Electronics|2156236000         |2.0242794E9             |19484            |1   |
|South |Office     |662524000          |6.060058E8              |6495             |2   |
|South |Accessories|113631000          |1.05018025E8            |6494             |3   |
|South |Networking |87669000           |8.2684125E7             |6494             |4   |
|West  |Electronics|2493348000         |2.2992404E9             |12985            |1   |
|West  |Accessories|6

In [35]:
sales_processed.repartition(2).write.mode("overwrite").parquet("final_challenge_output_parquet")
print("Pipeline complete. Output written safely to Parquet format.")

Pipeline complete. Output written safely to Parquet format.
